# JSMA: Jacobian and Gradients

This notebook follows the HTB Academy **Jacobian and Gradients** subsection. We build the reusable pieces needed by the Jacobian-based Saliency Map Attack (JSMA): one-class input gradients, the full Jacobian, target/competitor extraction, and search-space masking.

JSMA is targeted: it asks which input pixels can increase a chosen target logit while decreasing the other logits. The attack loop and saliency formula come in later subsections.

## Mathematical map

For class score `F_i(x)` and pixel `x_j`, the Jacobian entry is

`J_ij = partial F_i / partial x_j`

Read this aloud as: **J sub i-j equals the partial derivative of F sub i with respect to x sub j**. It means how much class `i`'s score changes when pixel `j` changes slightly. For `m` classes and `n` input features, `J` has shape `(m, n)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from htb_ai_library.core import set_reproducibility
from htb_ai_library.data import get_mnist_loaders
from htb_ai_library.models import SimpleLeNet
from htb_ai_library.training import train_model
from htb_ai_library.utils import save_model, load_model
from htb_ai_library.visualization import use_htb_style

use_htb_style()
set_reproducibility(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

The next cell loads the same LeNet-like MNIST model used by HTB. A cached checkpoint avoids retraining on every run. `eval()` disables training-only behavior such as dropout.

In [ ]:
train_loader, test_loader = get_mnist_loaders(batch_size=128)
model_path = output_dir / 'mnist_target.pth'
model = SimpleLeNet().to(device)

if model_path.exists():
    print(f'Loading existing model from {model_path}')
    model = load_model(model, model_path, device)
else:
    print('Training new model...')
    model = train_model(model, train_loader, test_loader, epochs=5, learning_rate=0.001, device=device)
    save_model(model, model_path)

model.eval()
print('Model ready for JSMA gradients')

## One class gradient

Autograd needs one scalar output. Selecting `logits[0, class_idx]` means we differentiate one image's one class score with respect to every input pixel. We use logits rather than softmax probabilities so target and competitor sensitivities remain independent.

In [ ]:
def compute_class_gradient(x, model, class_idx, wrt='logits'):
    """Return d(selected class score) / d(input) as a flat NumPy vector."""
    if x.shape[0] != 1:
        raise ValueError('compute_class_gradient expects batch size 1')

    x_grad = x.detach().clone().requires_grad_(True)
    logits = model(x_grad)

    if wrt == 'logits':
        scalar = logits[0, class_idx]
    elif wrt == 'probabilities':
        probs = F.softmax(logits, dim=1)
        scalar = probs[0, class_idx]
    else:
        raise ValueError("wrt must be 'logits' or 'probabilities'")

    scalar.backward()
    return x_grad.grad.detach().cpu().numpy().flatten().copy()

In [ ]:
# A deterministic four-feature toy model makes the gradient shape visible.
model_test = nn.Sequential(nn.Flatten(), nn.Linear(4, 2)).eval()
x_test = torch.tensor([[[[0.5, 0.3], [0.2, 0.8]]]])
grad_class0 = compute_class_gradient(x_test, model_test, 0)
print('Gradient shape:', grad_class0.shape)
print('Gradient for class 0:', grad_class0)

A positive gradient component says increasing that feature raises the selected score; a negative component says decreasing it may help. Flattening preserves channel-height-width order, so the vector index maps consistently back to the image.

In [ ]:
def compute_jacobian_matrix(x, model, num_classes=10, wrt='logits'):
    """Return a (num_classes, num_features) Jacobian for one image."""
    if x.shape[0] != 1:
        raise ValueError('compute_jacobian_matrix expects batch size 1')

    rows = [
        compute_class_gradient(x, model, class_idx, wrt)
        for class_idx in range(num_classes)
    ]
    return np.asarray(rows)

print('For MNIST:', 10 * 28 * 28, 'Jacobian values')
print('Expected shape:', (10, 784))

Each Jacobian row is one class gradient. The explicit batch-size check prevents accidentally computing a result that does not correspond to one image's per-sample sensitivities.

In [ ]:
def extract_target_gradient(jacobian, target_class):
    return jacobian[target_class].copy()


def extract_other_gradients(jacobian, target_class):
    total_grad = jacobian.sum(axis=0)
    return total_grad - jacobian[target_class]

J_toy = np.array([
    [0.2, 0.5, -0.1, 0.3],
    [-0.1, 0.2, 0.4, -0.2],
    [0.6, -0.3, 0.1, 0.5],
])
target = 2
alpha = extract_target_gradient(J_toy, target)
beta = extract_other_gradients(J_toy, target)
print('Target gradient alpha:', alpha)
print('Other-gradient sum beta:', beta)

Read `alpha` as the target class's sensitivity vector. Read `beta` as the combined sensitivity of every non-target class. In later saliency code, JSMA looks for directions where alpha is positive and beta is negative (or the reverse direction).

In [ ]:
def apply_search_mask(gradient, search_space):
    """Zero unavailable features without changing their indices."""
    return gradient * search_space

grad = np.array([0.5, -0.2, 0.8, 0.1, -0.4])
mask = np.array([True, False, True, False, True])
masked_grad = apply_search_mask(grad, mask)
print('Original gradient:', grad)
print('Search-space mask:', mask)
print('Masked gradient:  ', masked_grad)

Multiplication converts `True` to 1 and `False` to 0. We keep the vector length and therefore preserve pixel-to-index correspondence. This mask can exclude pixels that are already saturated at `clip_min`/`clip_max` or have already been consumed by the attack.

**Notebook boundary:** this subsection stops here. Saliency scoring and feature selection belong to the next HTB subsection.

## Saliency scoring

For feature `j`, `alpha_j` (say **alpha sub j**) is the target gradient and `beta_j` (say **beta sub j**) is the combined competitor gradient. For an increase, `alpha_j > 0` and `beta_j < 0`; the score is `|alpha_j| × |beta_j|` when valid, otherwise zero. We score increase and decrease directions separately because reversing a pixel reverses the useful sign pattern.

In [ ]:
def score_increase_saliency(target_grad, other_grad):
    increase_mask = (target_grad > 0) & (other_grad < 0)
    return target_grad * np.abs(other_grad) * increase_mask

def score_decrease_saliency(target_grad, other_grad):
    decrease_mask = (target_grad < 0) & (other_grad > 0)
    return np.abs(target_grad) * other_grad * decrease_mask

alpha = np.array([0.6, -0.3, 0.4, 0.1, -0.5, 0.2])
beta = np.array([-0.2, 0.4, -0.5, 0.3, 0.6, -0.1])
inc_scores = score_increase_saliency(alpha, beta)
dec_scores = score_decrease_saliency(alpha, beta)
print('Increase scores:', inc_scores)
print('Decrease scores:', dec_scores)

A zero score means a feature failed the sign test, not necessarily that its gradients were small. A valid feature must help the target and suppress competitors in the tested direction.

In [ ]:
def select_best_direction(inc_scores, dec_scores):
    max_inc_idx = int(np.argmax(inc_scores))
    max_dec_idx = int(np.argmax(dec_scores))
    max_inc_score = float(inc_scores[max_inc_idx])
    max_dec_score = float(dec_scores[max_dec_idx])
    if max_inc_score > max_dec_score:
        return max_inc_idx, max_inc_score, True
    return max_dec_idx, max_dec_score, False

pixel_idx, score, increase = select_best_direction(inc_scores, dec_scores)
direction = 'increase' if increase else 'decrease'
print(f'Selected pixel {pixel_idx}, score {score:.3f}, {direction}')

`argmax` is pronounced **argument max**: it returns the index where the largest value occurs. We compare the best increase against the best decrease. If every score is zero, the caller must stop instead of modifying the arbitrary index returned by `argmax`.

In [ ]:
def initialize_search_space(shape):
    num_features = int(np.prod(shape[1:]))
    return np.ones(num_features, dtype=bool)

def remove_saturated_pixels(search_space, x, clip_min=0.0, clip_max=1.0, epsilon=1e-6):
    x_flat = x.detach().cpu().numpy().flatten()
    saturated_min = x_flat <= clip_min + epsilon
    saturated_max = x_flat >= clip_max - epsilon
    saturated = saturated_min | saturated_max
    return search_space & ~saturated

shape = (1, 1, 28, 28)
search_space = initialize_search_space(shape)
print('Initial shape:', search_space.shape)
print('Initial available pixels:', search_space.sum())
x_toy = torch.tensor([[[[0.0, 0.3], [0.95, 1.0]]]])
toy_mask = np.ones(4, dtype=bool)
updated_mask = remove_saturated_pixels(toy_mask, x_toy)
print('Toy pixel values:', x_toy.flatten().numpy())
print('Remaining mask:', updated_mask)

`shape[1:]` ignores the batch dimension and multiplies channels, height, and width: for MNIST, `1 × 28 × 28 = 784`. A small epsilon (say **epsilon**, a tolerance) treats values extremely close to 0 or 1 as saturated. The next HTB section uses these scores and masks inside the single-pixel attack loop.

## Single-pixel attack utilities

This section connects the math to an actual image. Saliency returns a flat pixel index, so we flatten, modify one value, clamp it to the valid image range, and reshape back. We also need a target-success test and a probability-based progress metric.

In [ ]:
def apply_single_pixel_perturbation(x, pixel_idx, theta, increase, clip_min=0.0, clip_max=1.0):
    original_shape = x.shape
    x_flat = x.view(-1).clone()
    perturbation = theta if increase else -theta
    x_flat[pixel_idx] = torch.clamp(
        x_flat[pixel_idx] + perturbation, clip_min, clip_max
    )
    return x_flat.view(original_shape)

def check_target_reached(x, target_class, model):
    with torch.no_grad():
        prediction = int(model(x).argmax(dim=1).item())
    return prediction == target_class

def compute_confidence(x, target_class, model):
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)
        return float(probs[0, target_class].item())

`theta` (say **theta**) is the signed step size. `torch.clamp` enforces `[clip_min, clip_max]`. `check_target_reached` uses `argmax` (argument max) on logits, while `compute_confidence` applies softmax so the target's progress is expressed as a probability.

In [ ]:
# Select one correctly classified MNIST image and choose a different target.
for x_batch, y_batch in test_loader:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    with torch.no_grad():
        preds = model(x_batch).argmax(dim=1)
    for i in range(x_batch.size(0)):
        if preds[i].item() == y_batch[i].item():
            x = x_batch[i:i+1]
            original_class = int(y_batch[i].item())
            target_class = (original_class + 5) % 10
            break
    break

x_adv = x.clone().detach()
theta = 0.25
clip_min, clip_max = 0.0, 1.0
search_space = initialize_search_space(x.shape)
print(f'Sample: digit {original_class}, target {target_class}')
print(f'Initial target confidence: {compute_confidence(x_adv, target_class, model):.4f}')

## One complete iteration

The following cells intentionally expose the data flow instead of hiding it in one large function: Jacobian → alpha/beta → masked saliency → winning pixel → perturbation → updated mask → success check. This is the final boundary of this subsection; the next subsection packages the same steps into a reusable attack loop.

In [ ]:
jacobian = compute_jacobian_matrix(x_adv, model, num_classes=10, wrt='logits')
alpha = apply_search_mask(extract_target_gradient(jacobian, target_class), search_space)
beta = apply_search_mask(extract_other_gradients(jacobian, target_class), search_space)
inc_scores = score_increase_saliency(alpha, beta)
dec_scores = score_decrease_saliency(alpha, beta)
pixel_idx, saliency, increase = select_best_direction(inc_scores, dec_scores)

print('Jacobian shape:', jacobian.shape)
print('Valid increases:', int((inc_scores > 0).sum()))
print('Valid decreases:', int((dec_scores > 0).sum()))
print(f'Selected pixel {pixel_idx}, saliency {saliency:.6f}, {"increase" if increase else "decrease"}')

if saliency > 0:
    pixel_before = x_adv.view(-1)[pixel_idx].item()
    x_adv = apply_single_pixel_perturbation(
        x_adv, pixel_idx, theta, increase, clip_min, clip_max
    )
    pixel_after = x_adv.view(-1)[pixel_idx].item()
    search_space = remove_saturated_pixels(
        search_space, x_adv, clip_min, clip_max
    )
    print(f'Pixel value: {pixel_before:.4f} -> {pixel_after:.4f}')
    print('Target reached:', check_target_reached(x_adv, target_class, model))
    print('Target confidence:', f'{compute_confidence(x_adv, target_class, model):.4f}')
else:
    print('No valid saliency score; stop the attack.')

## Complete single-pixel attack loop

The loop is a greedy optimizer: at each iteration it chooses the currently best pixel and direction, makes that one irreversible change, and repeats. `theta` controls each step, `gamma` controls the maximum fraction of changed pixels, and `max_iter` prevents an endless run.

In [ ]:
# Reset the candidate and configure this experiment.
x_adv = x.clone().detach()
search_space = initialize_search_space(x.shape)
config = {
    'theta': 0.25,
    'gamma': 0.15,
    'max_iter': 100,
    'wrt': 'logits',
    'clip_min': 0.0,
    'clip_max': 1.0,
}
num_features = int(np.prod(x.shape[1:]))
max_pixels = int(config['gamma'] * num_features)
stats = {'iterations': [], 'pixels_modified': [],
         'target_confidence': [], 'saliency_scores': []}
pixels_modified = 0
print(f'Maximum changed pixels: {max_pixels} of {num_features}')

The budget equation is

`max_pixels = gamma × num_features`

Read aloud: **maximum pixels equals gamma times number of features**. For MNIST, `0.15 × 784 = 117` pixels. This is a hard limit on the count of modified features, an L-zero-style constraint.

In [ ]:
for iteration in range(config['max_iter']):
    # Cheap stopping checks come before the expensive Jacobian.
    if check_target_reached(x_adv, target_class, model):
        print(f'Target reached at iteration {iteration}')
        break
    if pixels_modified >= max_pixels:
        print('Pixel budget exhausted')
        break

    jacobian = compute_jacobian_matrix(
        x_adv, model, num_classes=10, wrt=config['wrt']
    )
    alpha = extract_target_gradient(jacobian, target_class)
    beta = extract_other_gradients(jacobian, target_class)
    alpha = apply_search_mask(alpha, search_space)
    beta = apply_search_mask(beta, search_space)

    inc_scores = score_increase_saliency(alpha, beta)
    dec_scores = score_decrease_saliency(alpha, beta)
    pixel_idx, saliency, increase = select_best_direction(
        inc_scores, dec_scores
    )
    if saliency <= 0:
        print('No valid pixels remain')
        break

    x_adv = apply_single_pixel_perturbation(
        x_adv, pixel_idx, config['theta'], increase,
        config['clip_min'], config['clip_max']
    )
    search_space[pixel_idx] = False
    search_space = remove_saturated_pixels(
        search_space, x_adv, config['clip_min'], config['clip_max']
    )

    pixels_modified += 1
    confidence = compute_confidence(x_adv, target_class, model)
    stats['iterations'].append(iteration)
    stats['pixels_modified'].append(pixels_modified)
    stats['target_confidence'].append(confidence)
    stats['saliency_scores'].append(saliency)

    if iteration % 5 == 0 or iteration < 3:
        print(f'{iteration:>3}  pixels={pixels_modified:>3}  '
              f'confidence={confidence:.4f}  saliency={saliency:.6f}')

The order is deliberate. A forward-only success check is cheap; a full Jacobian requires one backward pass per class. The loop therefore avoids calculating gradients after success or budget exhaustion. Each iteration masks the selected pixel immediately, then removes all pixels at either clipping boundary.

In [ ]:
final_success = check_target_reached(x_adv, target_class, model)
final_pred = int(model(x_adv).argmax(dim=1).item())
print('Final result:', 'SUCCESS' if final_success else 'FAILED')
print('Final prediction:', final_pred)
print(f'Pixels modified: {pixels_modified}/{max_pixels}')
print('Iterations recorded:', len(stats['iterations']))

A failed run is still informative. Falling saliency scores mean the greedy search is consuming the strongest currently available choices. Very low target confidence means the candidate never approached the decision boundary. This single-pixel baseline is intentionally limited; later HTB sections use pairwise selection and batch evaluation for stronger, more representative attacks.

## Batch evaluation

One sample cannot tell us whether a failure is typical. We now collect correctly classified examples, attack each independently, and summarize success rate, pixel counts, and perturbation norms. Each sample gets a fresh `x_adv` and search mask.

In [ ]:
target_count = 10
original_images, original_labels, target_labels = [], [], []

for x_batch, y_batch in test_loader:
    if len(original_images) >= target_count:
        break
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    with torch.no_grad():
        predictions = model(x_batch).argmax(dim=1)
    for i in range(x_batch.size(0)):
        if len(original_images) >= target_count:
            break
        if predictions[i].item() != y_batch[i].item():
            continue
        original_images.append(x_batch[i:i+1])
        original_class = int(y_batch[i].item())
        original_labels.append(original_class)
        target_labels.append((original_class + 5) % 10)

print(f'Collected {len(original_images)} correctly classified samples')

The target is offset by five classes using `(original + 5) % 10`, which wraps around after class 9. Keeping the image, original label, and target label in parallel lists means index `i` always refers to one complete attack case.

In [ ]:
results = {'adversarial': [], 'success': [],
          'pixels_modified': [], 'iterations': []}

for idx, (x_orig, target) in enumerate(zip(original_images, target_labels)):
    x_adv = x_orig.clone().detach()
    search_space = initialize_search_space(x_orig.shape)
    pixels_mod = 0
    max_pixels = int(config['gamma'] * np.prod(x_orig.shape[1:]))

    for iteration in range(config['max_iter']):
        if check_target_reached(x_adv, target, model) or pixels_mod >= max_pixels:
            break
        jacobian = compute_jacobian_matrix(x_adv, model, 10, config['wrt'])
        alpha = apply_search_mask(extract_target_gradient(jacobian, target), search_space)
        beta = apply_search_mask(extract_other_gradients(jacobian, target), search_space)
        inc = score_increase_saliency(alpha, beta)
        dec = score_decrease_saliency(alpha, beta)
        pixel_idx, saliency, increase = select_best_direction(inc, dec)
        if saliency <= 0:
            break
        x_adv = apply_single_pixel_perturbation(
            x_adv, pixel_idx, config['theta'], increase,
            config['clip_min'], config['clip_max']
        )
        search_space[pixel_idx] = False
        search_space = remove_saturated_pixels(
            search_space, x_adv, config['clip_min'], config['clip_max']
        )
        pixels_mod += 1

    success = check_target_reached(x_adv, target, model)
    results['adversarial'].append(x_adv)
    results['success'].append(success)
    results['pixels_modified'].append(pixels_mod)
    results['iterations'].append(iteration + 1)
    status = 'SUCCESS' if success else 'FAILED'
    print(f'{idx + 1:>2}: {original_labels[idx]} -> {target}  {status:<7} '
          f'pixels={pixels_mod:<3} iterations={iteration + 1}')

Every sample repeats the same greedy process independently. `iteration + 1` corrects for Python's zero-based loop counter. A run can stop because it succeeds, reaches its pixel budget, reaches `max_iter`, or has no valid saliency candidates.

In [ ]:
success_count = sum(results['success'])
total_samples = len(results['success'])
pixels = np.asarray(results['pixels_modified'])
print(f'Success rate: {success_count}/{total_samples} '
      f'({100 * success_count / total_samples:.1f}%)')
print(f'Mean pixels: {pixels.mean():.1f} +/- {pixels.std():.1f}')
print(f'Median pixels: {np.median(pixels):.1f}')
print(f'Pixel range: [{pixels.min()}, {pixels.max()}]')
print(f'Mean sparsity: {100 * pixels.mean() / 784:.2f}%')

The mean sparsity calculation is

`mean changed pixels / 784 × 100`

Read aloud: **mean changed pixels divided by seven hundred eighty-four times one hundred**. This expresses the average L-zero modification count as a percentage of the image.

In [ ]:
# Compare L0, L1, L2, and L-infinity for the first result.
if results['adversarial']:
    perturbation = (results['adversarial'][0] - original_images[0]).cpu().numpy()
    magnitude = np.abs(perturbation)
    print('L0 (pixels changed):', np.count_nonzero(perturbation))
    print('L1 (sum of changes):', f'{magnitude.sum():.4f}')
    print('L2 (Euclidean magnitude):', f'{np.linalg.norm(perturbation):.4f}')
    print('L-infinity (largest change):', f'{magnitude.max():.4f}')

The norms describe different aspects of the same perturbation:

- `L0`: how many pixels changed.
- `L1`: total absolute amount of change.
- `L2`: overall Euclidean magnitude.
- `L-infinity`: the largest change to any one pixel.

This section's main lesson is that single-pixel JSMA can succeed on many samples, but results vary substantially. The canonical pairwise method in later sections is designed to capture interactions that one-pixel greedy selection misses.

## Configuration experiments

The same attack can behave very differently depending on step size `theta` and feature budget `gamma`. We vary one parameter at a time while keeping the other settings fixed. Each experiment starts from a fresh image and search mask.

In [ ]:
theta_values = [0.10, 0.25, 0.50, 1.00]
theta_results = []

for theta_test in theta_values:
    x_test = original_images[0].clone().detach()
    mask_test = initialize_search_space(x_test.shape)
    pixels_mod = 0
    target = target_labels[0]
    for iteration in range(config['max_iter']):
        if check_target_reached(x_test, target, model) or pixels_mod >= int(config['gamma'] * 784):
            break
        jacobian = compute_jacobian_matrix(x_test, model, 10, config['wrt'])
        alpha = apply_search_mask(extract_target_gradient(jacobian, target), mask_test)
        beta = apply_search_mask(extract_other_gradients(jacobian, target), mask_test)
        inc = score_increase_saliency(alpha, beta)
        dec = score_decrease_saliency(alpha, beta)
        pixel_idx, saliency, increase = select_best_direction(inc, dec)
        if saliency <= 0:
            break
        x_test = apply_single_pixel_perturbation(x_test, pixel_idx, theta_test, increase)
        mask_test[pixel_idx] = False
        mask_test = remove_saturated_pixels(mask_test, x_test)
        pixels_mod += 1
    theta_results.append({
        'theta': theta_test,
        'iterations': iteration + 1,
        'pixels': pixels_mod,
        'success': check_target_reached(x_test, target, model),
    })

print('theta   iterations   pixels   result')
for row in theta_results:
    print(f"{row['theta']:<7.2f} {row['iterations']:<12} {row['pixels']:<8} {"SUCCESS" if row['success'] else "FAILED"}")

A larger `theta` reaches boundaries quickly and can cross the decision boundary with fewer iterations, but it produces more visible changes. A smaller `theta` is gentler, yet may not accumulate enough change before `max_iter`. This is a threshold-like trade-off, not a guarantee that larger is always better.

In [ ]:
gamma_values = [0.10, 0.15, 0.20, 0.30]
print('gamma   max pixels   percentage')
for gamma_test in gamma_values:
    max_pixels_test = int(gamma_test * 784)
    print(f'{gamma_test:<7.2f} {max_pixels_test:<12} {gamma_test * 100:.0f}%')

`gamma` is a hard sparsity budget, not a promise that the attack will use every allowed pixel. A tighter budget can stop an attack that needs more features; a looser budget gives more headroom but may not improve success if the greedy search is the real limitation.

The next HTB section introduces **Pairwise Saliency**, where two pixels are selected together to capture feature interactions that this single-pixel baseline cannot see.

## Pairwise saliency

Single-pixel JSMA can miss useful feature interactions. For a pair `(p, q)`, the combined sensitivities are `alpha_pq = alpha_p + alpha_q` and `beta_pq = beta_p + beta_q`. Read these aloud as **alpha sub p-q equals alpha sub p plus alpha sub q**, and similarly for beta. We apply the same sign constraints and magnitude product to the pair sums.

In [ ]:
def prune_candidates(alpha, beta, search_space, top_k):
    valid = np.where(search_space)[0]
    if valid.size < 2 or top_k is None or valid.size <= top_k:
        return valid
    preliminary = np.abs(alpha[valid]) * np.abs(beta[valid])
    return valid[np.argsort(-preliminary)[:top_k]]

def evaluate_pairs(alpha, beta, valid, direction):
    best_p, best_q, best_score = -1, -1, 0.0
    for i in range(valid.size):
        p = valid[i]
        for j in range(i + 1, valid.size):
            q = valid[j]
            alpha_pair = alpha[p] + alpha[q]
            beta_pair = beta[p] + beta[q]
            if direction == 'increase':
                if alpha_pair <= 0 or beta_pair >= 0:
                    continue
                score = alpha_pair * abs(beta_pair)
            else:
                if alpha_pair >= 0 or beta_pair <= 0:
                    continue
                score = abs(alpha_pair) * beta_pair
            if score > best_score:
                best_p, best_q, best_score = int(p), int(q), float(score)
    return best_p, best_q, best_score

def compute_pairwise_saliency(alpha, beta, search_space, direction='increase', top_k=None):
    valid = prune_candidates(alpha, beta, search_space, top_k)
    if valid.size < 2:
        return -1, -1, 0.0
    return evaluate_pairs(alpha, beta, valid, direction)

def apply_pair_perturbation(x, p, q, theta, increase, clip_min=0.0, clip_max=1.0):
    original_shape = x.shape
    x_flat = x.view(-1).clone()
    step = theta if increase else -theta
    x_flat[p] = torch.clamp(x_flat[p] + step, clip_min, clip_max)
    x_flat[q] = torch.clamp(x_flat[q] + step, clip_min, clip_max)
    return x_flat.view(original_shape)

Without pruning, `n` valid pixels produce

`n choose 2 = n × (n - 1) / 2`

Read aloud: **n choose two equals n times n minus one divided by two**. With 784 pixels this is 306,936 pairs. `top_k=128` reduces that to 8,128 pairs. Pruning is a speed/optimality trade-off: it may miss the global best pair, but makes the search feasible.

In [ ]:
# A small deterministic example showing pair sums and direction choice.
alpha_demo = np.array([0.6, 0.5, -0.4, 0.1])
beta_demo = np.array([-0.3, -0.4, 0.6, 0.2])
mask_demo = np.ones(4, dtype=bool)
p, q, score = compute_pairwise_saliency(
    alpha_demo, beta_demo, mask_demo, 'increase', top_k=None
)
print(f'Best increasing pair: ({p}, {q}), score={score:.3f}')
print(f'Pair alpha: {alpha_demo[p] + alpha_demo[q]:.3f}')
print(f'Pair beta:  {beta_demo[p] + beta_demo[q]:.3f}')

## Complete pairwise attack loop

Pairwise JSMA now selects two pixels at a time. The pair budget is still counted in pixels, so `pixels_modified` increases by 2 per iteration. `top_k` limits the candidate set before pair enumeration, keeping the quadratic search manageable.

In [ ]:
x_adv = x.clone().detach()
search_space = initialize_search_space(x.shape)
pair_config = {
    'theta': 1.0, 'gamma': 0.15, 'max_iter': 90,
    'wrt': 'logits', 'clip_min': 0.0, 'clip_max': 1.0,
    'top_k': 128,
}
num_features = int(np.prod(x.shape[1:]))
max_pixels = int(pair_config['gamma'] * num_features)
pixels_modified = 0
pair_stats = {'iterations': [], 'pixels_modified': [],
              'target_confidence': [], 'pair_scores': []}

for iteration in range(pair_config['max_iter']):
    if check_target_reached(x_adv, target_class, model):
        print(f'Target reached at iteration {iteration}')
        break
    if pixels_modified + 2 > max_pixels:
        print('Pairwise pixel budget exhausted')
        break

    jacobian = compute_jacobian_matrix(
        x_adv, model, 10, pair_config['wrt']
    )
    alpha = extract_target_gradient(jacobian, target_class)
    beta = extract_other_gradients(jacobian, target_class)
    p_inc, q_inc, score_inc = compute_pairwise_saliency(
        alpha, beta, search_space, 'increase', pair_config['top_k']
    )
    p_dec, q_dec, score_dec = compute_pairwise_saliency(
        alpha, beta, search_space, 'decrease', pair_config['top_k']
    )

    if max(score_inc, score_dec) <= 0:
        print('No valid pair remains')
        break
    if score_inc >= score_dec:
        p, q, score, increase = p_inc, q_inc, score_inc, True
    else:
        p, q, score, increase = p_dec, q_dec, score_dec, False
    if p < 0 or q < 0:
        print('Invalid pair sentinel returned')
        break

    x_adv = apply_pair_perturbation(
        x_adv, p, q, pair_config['theta'], increase,
        pair_config['clip_min'], pair_config['clip_max']
    )
    search_space[p] = False
    search_space[q] = False
    search_space = remove_saturated_pixels(
        search_space, x_adv, pair_config['clip_min'], pair_config['clip_max']
    )
    pixels_modified += 2
    confidence = compute_confidence(x_adv, target_class, model)
    pair_stats['iterations'].append(iteration)
    pair_stats['pixels_modified'].append(pixels_modified)
    pair_stats['target_confidence'].append(confidence)
    pair_stats['pair_scores'].append(score)
    if iteration % 3 == 0 or iteration < 3:
        print(f'{iteration:>3}  pixels={pixels_modified:>3}  '
              f'confidence={confidence:.4f}  score={score:.6f}')

print('Final success:', check_target_reached(x_adv, target_class, model))
print('Final prediction:', int(model(x_adv).argmax(dim=1).item()))
print(f'Pixels modified: {pixels_modified}/{max_pixels}')

The loop is greedy and irreversible, just like the single-pixel version, but each decision evaluates a coordinated pair. The condition `pixels_modified + 2 > max_pixels` prevents exceeding the budget by half a pair. Pairwise selection often reaches the target with fewer iterations and fewer total changed pixels because it can exploit feature synergy.

## Pairwise batch evaluation

Repeat the pairwise attack independently for every sample. Each sample gets a fresh adversarial tensor and search-space mask; parallel result lists keep metrics aligned by sample index.

In [ ]:
results_pairs = {'adversarial': [], 'success': [], 'pixels_modified': [], 'iterations': []}
print('Running pairwise attacks...')
print(f"{'#':<4} {'Orig→Tgt':<10} {'Result':<8} {'Pixels':<8} {'Iters':<8}")
print('=' * 46)

In [ ]:
for idx in range(len(original_images)):
    x_i = original_images[idx]
    orig_class_i = int(original_labels[idx])
    target_class_i = int(target_labels[idx])
    x_adv_i = x_i.clone().detach()
    search_space_i = initialize_search_space(x_i.shape)
    pixels_mod = 0
    max_pixels_i = int(pair_config['gamma'] * np.prod(x_i.shape[1:]))

    for iteration_i in range(pair_config['max_iter']):
        if check_target_reached(x_adv_i, target_class_i, model) or pixels_mod + 2 > max_pixels_i:
            break
        jacobian_i = compute_jacobian_matrix(x_adv_i, model, 10, pair_config['wrt'])
        alpha_i = extract_target_gradient(jacobian_i, target_class_i)
        beta_i = extract_other_gradients(jacobian_i, target_class_i)
        p_inc, q_inc, score_inc = compute_pairwise_saliency(alpha_i, beta_i, search_space_i, 'increase', pair_config['top_k'])
        p_dec, q_dec, score_dec = compute_pairwise_saliency(alpha_i, beta_i, search_space_i, 'decrease', pair_config['top_k'])
        if max(score_inc, score_dec) <= 0.0:
            break
        if score_inc >= score_dec:
            p, q, score, increase = p_inc, q_inc, score_inc, True
        else:
            p, q, score, increase = p_dec, q_dec, score_dec, False
        if p < 0 or q < 0:
            break
        x_adv_i = apply_pair_perturbation(x_adv_i, p, q, pair_config['theta'], increase, pair_config['clip_min'], pair_config['clip_max'])
        search_space_i[p] = False
        search_space_i[q] = False
        search_space_i = remove_saturated_pixels(search_space_i, x_adv_i, pair_config['clip_min'], pair_config['clip_max'])
        pixels_mod += 2

    success_i = check_target_reached(x_adv_i, target_class_i, model)
    results_pairs['adversarial'].append(x_adv_i)
    results_pairs['success'].append(success_i)
    results_pairs['pixels_modified'].append(pixels_mod)
    results_pairs['iterations'].append(iteration_i + 1)
    status = '✓' if success_i else '✗'
    print(f'{idx + 1:<4} {orig_class_i}→{target_class_i:<8} {status:<8} {pixels_mod:<8} {iteration_i + 1:<8}')

print('=' * 46)

The pairwise budget is counted in pixels, not loop iterations: one iteration changes two pixels, so the guard `pixels_mod + 2 > max_pixels_i` prevents overshooting. `success_i` is checked again after the loop so every termination reason is evaluated consistently.

In [ ]:
print('\nEfficiency comparison')
for idx in range(len(original_images)):
    print(f'Sample {idx + 1}: single iters={results["iterations"][idx]}, pairwise iters={results_pairs["iterations"][idx]}, single pixels={results["pixels_modified"][idx]}, pairwise pixels={results_pairs["pixels_modified"][idx]}')
single_success_rate = 100 * np.mean(results['success'])
pair_success_rate = 100 * np.mean(results_pairs['success'])
mean_single_iters = np.mean(results['iterations'])
mean_pair_iters = np.mean(results_pairs['iterations'])
mean_single_pixels = np.mean(results['pixels_modified'])
mean_pair_pixels = np.mean(results_pairs['pixels_modified'])
print(f'Success rate: single={single_success_rate:.1f}%, pairwise={pair_success_rate:.1f}%')
print(f'Mean iterations: single={mean_single_iters:.1f}, pairwise={mean_pair_iters:.1f}')
print(f'Mean changed pixels: single={mean_single_pixels:.1f}, pairwise={mean_pair_pixels:.1f}')
print(f'Iteration reduction: {100 * (mean_single_iters - mean_pair_iters) / mean_single_iters:.1f}%')
print(f'Pixel reduction: {100 * (mean_single_pixels - mean_pair_pixels) / mean_single_pixels:.1f}%')

## Pairwise analysis: synergy and computational trade-offs

A pair's measured impact can be compared with the sum of its individual impacts. Read `synergy = pair impact − expected impact` aloud as ‘synergy equals pair impact minus expected impact.’ Positive synergy means the combination helps more than additivity predicts; negative synergy means interference.

In [ ]:
# Test the first selected pair in isolation (not in the full attack context).
p, q = (352, 389)
x_test = original_images[0].clone().detach()
target_i = int(target_labels[0])
baseline_conf = compute_confidence(x_test, target_i, model)
x_p = apply_single_pixel_perturbation(x_test, p, 1.0, True, 0.0, 1.0)
x_q = apply_single_pixel_perturbation(x_test, q, 1.0, True, 0.0, 1.0)
x_pq = apply_pair_perturbation(x_test, p, q, 1.0, True, 0.0, 1.0)
impact_p = compute_confidence(x_p, target_i, model) - baseline_conf
impact_q = compute_confidence(x_q, target_i, model) - baseline_conf
expected_impact = impact_p + impact_q
impact_pq = compute_confidence(x_pq, target_i, model) - baseline_conf
synergy = impact_pq - expected_impact
synergy_pct = 100 * synergy / expected_impact if expected_impact > 0 else 0.0
print(f'Pair ({p}, {q}) individual impacts: {impact_p:.6f}, {impact_q:.6f}')
print(f'Expected additive impact: {expected_impact:.6f}')
print(f'Actual pair impact: {impact_pq:.6f}')
print(f'Synergy: {synergy:.6f} ({synergy_pct:+.2f}%)')

A zero result does not prove that pairwise JSMA is useless: this test isolates one pair on the original image, while the real attack uses pairs sequentially after the model representation has already changed. Neural-network nonlinearities can make interactions strongly context-dependent.

In [ ]:
import time
# Benchmark scoring only; Jacobian work is measured separately in practice.
alpha_test, beta_test = alpha, beta
search_space_test = np.ones_like(alpha_test, dtype=bool)
start = time.perf_counter()
for _ in range(100):
    _ = np.argmax(score_increase_saliency(alpha_test, beta_test))
single_time = (time.perf_counter() - start) / 100
start = time.perf_counter()
for _ in range(100):
    _ = compute_pairwise_saliency(alpha_test, beta_test, search_space_test, 'increase', top_k=128)
pair_time = (time.perf_counter() - start) / 100
print(f'Single saliency: {single_time * 1000:.3f} ms')
print(f'Pairwise saliency: {pair_time * 1000:.3f} ms')
print(f'Pairwise/single ratio: {pair_time / max(single_time, 1e-12):.1f}x')
jacobian_time = 15.0
print(f'Estimated iteration cost: single={jacobian_time + single_time * 1000:.2f} ms, pairwise={jacobian_time + pair_time * 1000:.2f} ms')

Pair enumeration is O(n²)—read ‘order n squared’—because `n` candidates can form `n choose 2` pairs. `top_k=128` limits this to at most `128 choose 2 = 8,128` pairs per direction. Smaller `top_k` is faster but can discard the best pair; `top_k=None` searches all pairs but costs more. A larger `theta` moves pixels farther per step, while `gamma` limits the total fraction of changed pixels.